## Scenario - 3: Multiple data scientist working on multiple ML models

#### MLflow setup
- Tracking server: Yes, remote server (EC2)
- Backend store: postgres database
- Artifacts store: S3 bucket

The experiments can be explored by accessing the remote server

The example uses AWS to host a remote server. In order to run the example you'll need an AWS account.

In [1]:
import mlflow
import os

os.environ.pop("AWS_PROFILE", None)

TRACKING_SERVER_HOST = "ec2-13-235-13-158.ap-south-1.compute.amazonaws.com"
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [2]:
mlflow.get_tracking_uri()

'http://ec2-13-235-13-158.ap-south-1.compute.amazonaws.com:5000'

In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlops-mlflow-artifacts-s3-bucket/1', creation_time=1781023136518, experiment_id='1', last_update_time=1781023136518, lifecycle_stage='active', name='iris-classification', tags={}>,
 <Experiment: artifact_location='s3://mlops-mlflow-artifacts-s3-bucket/0', creation_time=1781001580339, experiment_id='0', last_update_time=1781001580339, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from mlflow.models.signature import infer_signature

mlflow.set_experiment("iris-classification")

with mlflow.start_run():
    
    mlflow.set_tag("Developer Name", "megh")
    mlflow.set_tag("model", "logistic_regression")
    
    X, y = load_iris(return_X_y=True)
    
    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)
    
    logistic_regression = LogisticRegression(**params).fit(X, y)
    y_pred = logistic_regression.predict(X)
    
    # Infer model signature and set a small input example to avoid the warning
    signature = infer_signature(X, y_pred)
    input_example = X[:3]
    mlflow.sklearn.log_model(logistic_regression, "model", signature=signature, input_example=input_example)
    
    print(f"default artifact location: {mlflow.get_artifact_uri()}")
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))


2026/06/09 17:42:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/home/codespace/anaconda3/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


default artifact location: s3://mlops-mlflow-artifacts-s3-bucket/1/9ee7406ccba342e7abbe6d33a87f29c9/artifacts
🏃 View run upset-auk-56 at: http://ec2-13-235-13-158.ap-south-1.compute.amazonaws.com:5000/#/experiments/1/runs/9ee7406ccba342e7abbe6d33a87f29c9
🧪 View experiment at: http://ec2-13-235-13-158.ap-south-1.compute.amazonaws.com:5000/#/experiments/1


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlops-mlflow-artifacts-s3-bucket/1', creation_time=1781023136518, experiment_id='1', last_update_time=1781023136518, lifecycle_stage='active', name='iris-classification', tags={}>,
 <Experiment: artifact_location='s3://mlops-mlflow-artifacts-s3-bucket/0', creation_time=1781001580339, experiment_id='0', last_update_time=1781001580339, lifecycle_stage='active', name='Default', tags={}>]

#### Interacting with MLflow model registry

In [6]:
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException

client = MlflowClient()

In [7]:
try:
    client.search_registered_models()
except MlflowException as e:
    print(e)

In [8]:
run_id = mlflow.search_runs(experiment_names=["iris-classification"], filter_string="metrics.accuracy > 0.9")["run_id"][0]
mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="iris-classification-model"
)

Successfully registered model 'iris-classification-model'.
2026/06/09 17:43:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 9ee7406ccba342e7abbe6d33a87f29c9 has no artifacts at artifact path 'model', registering model based on models:/m-ac8f84a2089f4fb8831ea96e14429323 instead
2026/06/09 17:43:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classification-model, version 1
Created version '1' of model 'iris-classification-model'.


<ModelVersion: aliases=[], creation_timestamp=1781026984338, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1781026984338, metrics=None, model_id=None, name='iris-classification-model', params=None, run_id='9ee7406ccba342e7abbe6d33a87f29c9', run_link='', source='models:/m-ac8f84a2089f4fb8831ea96e14429323', status='READY', status_message=None, tags={}, user_id='', version='1'>